[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_MusicGen.ipynb)


# 音楽生成デモ（MusicGen）

テキスト（日本語／英語）から短い音楽クリップを生成するデモです．  
日本語プロンプトは [Gemini API](https://ai.google.dev/) で英語に翻訳し，[Meta MusicGen](https://huggingface.co/facebook/musicgen-small) で音声を生成します．

**実行環境**: Google Colab（ランタイム → GPU: T4 推奨）

**必要なトークン**: `tokens.json` の **`gemini`** キー（Gemini API キー）．上部のセルでダウンロード・復号してください．

## セルの進め方
1. **tokens.json の準備**（ダウンロードと復号）
2. **設定**（モデル・生成秒数など）
3. **ライブラリのインストール**
4. **ライブラリの読み込み・モデル準備**
5. **Gradio の起動**

> MusicGen のライセンスは **CC-BY-NC 4.0**（非営利・教育デモ向け）です．


## tokens.json の準備（必須）

このデモでは **Gemini API キー**（`tokens.json` の `gemini` キー）が必要です．  
次のセルを実行し，暗号化ファイルをダウンロードして復号してください（パスフレーズを求められたら入力します）．


In [ ]:
!wget https://github.com/yryo1005/cursor_template/releases/download/v1.0.0/tokens.json.enc

!openssl enc -d -aes-256-cbc -pbkdf2 -iter 100000 \
  -in tokens.json.enc \
  -out tokens.json


## 0. 設定

- 生成が遅い／粗い場合は `DURATION_SEC` や `MODEL_ID` を調整してください．
- `musicgen-small` は T4 向け．品質を上げたい場合は `facebook/musicgen-medium`（VRAM・時間が増えます）．
- 変更後は **初期化セル** と **Gradio 起動セル** を再実行してください．


In [ ]:
# MusicGen の生成設定（T4 / 14GB VRAM 向け）
# small ≈ 300M / medium ≈ 1.5B（medium は品質↑・時間↑）
MODEL_ID = "facebook/musicgen-small"
DURATION_SEC = 8  # 生成する秒数（長いほど時間がかかる．T4 では 5〜10 程度が無難）
GUIDANCE_SCALE = 3.0  # プロンプトへの追従度（大きいほど指定に寄りやすい）
GEMINI_MODEL = "gemini-2.5-flash"  # 日本語→英語翻訳用

print(f"MODEL_ID = {MODEL_ID}")
print(f"DURATION_SEC = {DURATION_SEC}")
print(f"GUIDANCE_SCALE = {GUIDANCE_SCALE}")
print(f"GEMINI_MODEL = {GEMINI_MODEL}")


## 1. ライブラリのインストール


In [ ]:
# Colab 標準の torch / transformers / gradio を利用
# MusicGen 用に transformers を更新．Gemini 翻訳用に google-genai も念のため
!pip install -q -U "transformers>=4.40.0" "google-genai>=1.0.0"


## 2. ライブラリの読み込み，変数のインスタンス化

`tokens.json`（`gemini` キー）を読み込み，翻訳クライアントと MusicGen を準備します．  
初回はモデルのダウンロードに数分かかります．


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import gradio as gr
import numpy as np
import torch
from google import genai
from tqdm.auto import tqdm
from transformers import AutoProcessor, MusicgenForConditionalGeneration

# ------------------------------------------------------------
# 定数・サンプルプロンプト
# ------------------------------------------------------------
TOKENS_PATH = Path("tokens.json")

# MusicGen は約 50 フレーム/秒でトークンを生成する
FRAMES_PER_SECOND = 50

# Gradio Examples 用（日本語・英語を混ぜる）
SAMPLE_PROMPTS: list[str] = [
    "明るく爽やかなアコースティックギターのポップス",
    "夜の街を走るようなシンセウェーブ，レトロフューチャー",
    "a calm lo-fi hip hop beat with soft piano and rain",
    "壮大なオーケストラ，映画のクライマックス風",
    "upbeat electronic dance music with punchy drums",
    "静かなジャズピアノのトリオ，カフェで流れる雰囲気",
]


def load_tokens(path: Path = TOKENS_PATH) -> dict:
    """API キーを tokens.json から読み込む．

    Args:
        path (Path): トークンファイルのパス

    Returns:
        dict: キー名とトークン文字列の辞書

    Raises:
        FileNotFoundError: ファイルが無い場合
        KeyError: gemini キーが無い場合
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{path.resolve()} が見つかりません．"
            '{"gemini": "..."} を含む tokens.json を配置してください．'
        )
    with path.open(encoding="utf-8") as f:
        tokens = json.load(f)
    if "gemini" not in tokens or not tokens["gemini"]:
        raise KeyError("tokens.json に gemini キーがありません．")
    return tokens


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（非常に時間がかかります）．")
    return "cpu"


def contains_japanese(text: str) -> bool:
    """文字列に日本語（ひらがな・カタカナ・漢字）が含まれるか判定する．

    Args:
        text (str): 判定対象の文字列

    Returns:
        bool: 日本語を含む場合 True
    """
    return bool(re.search(r"[\u3040-\u30ff\u3400-\u9fff]", text))


def translate_prompt_to_english(prompt: str, client: genai.Client, model: str) -> str:
    """日本語プロンプトを MusicGen 向け英語に翻訳する．

    英語のみの場合はそのまま返す．

    Args:
        prompt (str): ユーザー入力プロンプト
        client (genai.Client): Gemini クライアント
        model (str): Gemini モデル名

    Returns:
        str: 英語プロンプト
    """
    if not contains_japanese(prompt):
        return prompt.strip()

    instruction = (
        "Translate the following music-generation prompt into natural English "
        "suitable for MusicGen (text-to-music). "
        "Keep genre, mood, instruments, and tempo cues. "
        "Output ONLY the English prompt, with no quotes or explanation.\n\n"
        f"Prompt:\n{prompt}"
    )
    response = client.models.generate_content(model=model, contents=instruction)
    english = (response.text or "").strip().strip('"').strip("'")
    if not english:
        raise RuntimeError("Gemini から翻訳結果を取得できませんでした．")
    return english


def duration_to_max_new_tokens(duration_sec: float) -> int:
    """秒数を MusicGen の max_new_tokens に変換する．

    Args:
        duration_sec (float): 生成したい秒数

    Returns:
        int: 生成トークン数（おおよそ duration_sec * 50）
    """
    return max(50, int(round(float(duration_sec) * FRAMES_PER_SECOND)))


def load_musicgen(
    model_id: str,
    device: str,
) -> tuple[AutoProcessor, MusicgenForConditionalGeneration, int]:
    """MusicGen のプロセッサとモデルを読み込む．

    Args:
        model_id (str): Hugging Face モデル ID
        device (str): "cuda" または "cpu"

    Returns:
        tuple[AutoProcessor, MusicgenForConditionalGeneration, int]:
            (プロセッサ, モデル, サンプリングレート Hz)
    """
    dtype = torch.float16 if device == "cuda" else torch.float32
    print(f"モデルを読み込み中: {model_id} (dtype={dtype})")
    processor = AutoProcessor.from_pretrained(model_id)
    model = MusicgenForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype,
    )
    model = model.to(device)
    model.eval()
    sampling_rate = int(model.config.audio_encoder.sampling_rate)
    print(f"sampling_rate = {sampling_rate} Hz")
    return processor, model, sampling_rate


def generate_music(
    prompt: str,
    duration_sec: float | None = None,
    seed: int | None = None,
) -> tuple[tuple[int, np.ndarray] | None, str]:
    """プロンプトから音楽を生成し，使用した英語プロンプトも返す．

    Args:
        prompt (str): 日本語または英語のプロンプト
        duration_sec (float | None): 生成秒数．None なら設定値 DURATION_SEC
        seed (int | None): 乱数シード．None または負値ならランダム

    Returns:
        tuple[tuple[int, np.ndarray] | None, str]:
            ((サンプリングレート, 波形 float32 shape=(samples,)), 英語プロンプト／メッセージ)
            失敗時は (None, エラーメッセージ)
    """
    prompt = (prompt or "").strip()
    if not prompt:
        return None, "プロンプトを入力してください．"

    sec = float(DURATION_SEC if duration_sec is None else duration_sec)
    sec = float(np.clip(sec, 2.0, 15.0))
    max_new_tokens = duration_to_max_new_tokens(sec)

    english_prompt = translate_prompt_to_english(prompt, gemini_client, GEMINI_MODEL)

    if seed is not None and int(seed) >= 0:
        torch.manual_seed(int(seed))
        if device == "cuda":
            torch.cuda.manual_seed_all(int(seed))

    inputs = processor(
        text=[english_prompt],
        padding=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 生成は時間がかかるため tqdm で 1 ステップ可視化（本体の進捗は transformers 側）
    with torch.inference_mode():
        for _ in tqdm(range(1), desc=f"音楽生成 (~{sec:.0f}s)", leave=False):
            audio_values = musicgen_model.generate(
                **inputs,
                do_sample=True,
                guidance_scale=float(GUIDANCE_SCALE),
                max_new_tokens=max_new_tokens,
            )

    # audio_values: (batch, channels, samples) → モノラル波形
    waveform = audio_values[0, 0].detach().float().cpu().numpy()
    waveform = np.clip(waveform, -1.0, 1.0).astype(np.float32)

    note = english_prompt
    if contains_japanese(prompt):
        note = f"【翻訳後】{english_prompt}"
    note = f"{note}\n（約 {sec:.0f} 秒 / {max_new_tokens} tokens）"
    return (sampling_rate, waveform), note


def build_demo() -> gr.Blocks:
    """Gradio UI を構築する．

    Returns:
        gr.Blocks: デモ用 UI
    """
    with gr.Blocks(title="音楽生成デモ（MusicGen）") as demo:
        with gr.Row():
            with gr.Column(scale=1):
                prompt_in = gr.Textbox(
                    label="プロンプト",
                    lines=3,
                    placeholder="例: 明るく爽やかなアコースティックギターのポップス",
                )
                duration_in = gr.Slider(
                    label="生成秒数",
                    minimum=2,
                    maximum=15,
                    step=1,
                    value=float(DURATION_SEC),
                )
                seed_in = gr.Number(
                    label="シード（同じ曲調を再現したいとき．-1 でランダム）",
                    value=-1,
                    precision=0,
                )
                run_btn = gr.Button("音楽を生成", variant="primary")
            with gr.Column(scale=1):
                audio_out = gr.Audio(label="生成した音楽", type="numpy")
                prompt_out = gr.Textbox(
                    label="実際に使った英語プロンプト",
                    lines=3,
                    interactive=False,
                )

        examples = [[p, float(DURATION_SEC), -1] for p in SAMPLE_PROMPTS]
        gr.Examples(
            examples=examples,
            inputs=[prompt_in, duration_in, seed_in],
            label="サンプルプロンプト（クリックして入力欄に入れる）",
        )

        run_btn.click(
            fn=generate_music,
            inputs=[prompt_in, duration_in, seed_in],
            outputs=[audio_out, prompt_out],
        )

    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
tokens = load_tokens()
gemini_client = genai.Client(api_key=tokens["gemini"])
device = resolve_device()

for _ in tqdm(range(1), desc="MusicGen 準備", leave=False):
    processor, musicgen_model, sampling_rate = load_musicgen(MODEL_ID, device)

print("準備完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行


In [ ]:
demo = build_demo()
demo.launch(share=True)
